# Transfer Learning CIFAR10

* Train a simple convnet on the CIFAR dataset the first 5 output classes [0..4].
* Freeze convolutional layers and fine-tune dense layers for the last 5 ouput classes [5..9].


In [0]:
import tensorflow as tf
import keras

import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.utils import np_utils
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from keras.constraints import maxnorm

### 1. Import CIFAR10 data and create 2 datasets with one dataset having classes from 0 to 4 and other having classes from 5 to 9 

In [0]:
from keras.datasets import cifar10

In [0]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [0]:
X_train_04 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
X_train_59 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

X_test_04 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
X_test_59 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [0]:
y_train_04 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
y_train_59 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

y_test_04 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
y_test_59 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [0]:
np.unique(y_test_04)

array([0, 1, 2, 3, 4])

In [0]:
np.unique(y_test_59)

array([5, 6, 7, 8, 9])

### 2. Use One-hot encoding to divide y_train and y_test into required no of output classes

In [0]:
num_class_04 = len(np.unique(y_train_04))

y_train_04 = tf.keras.utils.to_categorical(y_train_04, num_classes=num_class_04)
y_test_04 = tf.keras.utils.to_categorical(y_test_04, num_classes=num_class_04)

### 3. Build a sequential neural network model which can classify the classes 0 to 4 of CIFAR10 dataset with at least 80% accuracy on test data

In [0]:
# normalize inputs from 0-255 to 0.0-1.0
X_train_04 = X_train_04.astype('float32')
X_test_04 = X_test_04.astype('float32')

X_train_04 = X_train_04 / 255.0
X_test_04 = X_test_04 / 255.0

In [0]:
# initialize model
model = Sequential()

# add input layer with input_size(32,32,3)
model.add(Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu', padding='same'))

# create Conv 2D layers , along with DropoUts and MaxPooling to avoid overfit
model.add(Dropout(0.2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
# Flatten output
model.add(Flatten())
model.add(Dropout(0.2))
# add Dense layers
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.2))

# add Output layer
model.add(Dense(num_class_04, activation='softmax'))

In [0]:
model.compile(optimizer='sgd', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_14 (Conv2D)           (None, 32, 32, 32)        896       
_________________________________________________________________
dropout_11 (Dropout)         (None, 32, 32, 32)        0         
_________________________________________________________________
conv2d_15 (Conv2D)           (None, 32, 32, 32)        9248      
_________________________________________________________________
max_pooling2d_7 (MaxPooling2 (None, 16, 16, 32)        0         
_________________________________________________________________
conv2d_16 (Conv2D)           (None, 16, 16, 64)        18496     
_________________________________________________________________
dropout_12 (Dropout)         (None, 16, 16, 64)        0         
_________________________________________________________________
conv2d_17 (Conv2D)           (None, 16, 16, 64)       

In [0]:
epochs=10
batch_size = 32

model.fit(X_train_04, y_train_04, validation_data=(X_test_04, y_test_04), batch_size=batch_size, epochs=epochs)

Train on 25000 samples, validate on 5000 samples
Epoch 1/10
25000/25000 [==============================] - 127s 5ms/sample - loss: 1.5314 - accuracy: 0.3050 - val_loss: 1.3826 - val_accuracy: 0.4246
Epoch 2/10
25000/25000 [==============================] - 130s 5ms/sample - loss: 1.3586 - accuracy: 0.4283 - val_loss: 1.3072 - val_accuracy: 0.4716
Epoch 3/10
25000/25000 [==============================] - 130s 5ms/sample - loss: 1.2324 - accuracy: 0.4832 - val_loss: 1.1383 - val_accuracy: 0.5388
Epoch 4/10
25000/25000 [==============================] - 132s 5ms/sample - loss: 1.1096 - accuracy: 0.5409 - val_loss: 1.0714 - val_accuracy: 0.5678
Epoch 5/10
25000/25000 [==============================] - 127s 5ms/sample - loss: 1.0285 - accuracy: 0.5792 - val_loss: 1.1676 - val_accuracy: 0.5162
Epoch 6/10
25000/25000 [==============================] - 132s 5ms/sample - loss: 0.9648 - accuracy: 0.6111 - val_loss: 0.9070 - val_accuracy: 0.6376
Epoch 7/10
25000/25000 [===========================

In [0]:
score = model.evaluate(X_test_04, y_test_04)
print('\n', 'Test accuracy for Model :', score[1]*100)

5000/5000 [==============================] - 3s 696us/sample - loss: 0.8190 - accuracy: 0.6728

 Test accuracy for Model : 67.28000044822693


### 4. In the model which was built above (for classification of classes 0-4 in CIFAR10), make only the dense layers to be trainable and conv layers to be non-trainable

In [0]:
#Set pre-trained model layers to not trainable
for layer in model.layers[:-5]:
    layer.trainable = False

In [0]:
for layer in model.layers:
    print(layer, layer.trainable)

<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16E3A470> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000001FA16E3A6A0> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16E3AE48> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x000001FA16E3ABE0> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16E6DD68> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000001FA16E90908> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16E9C828> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x000001FA16E9CA58> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16EE8898> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000001FA16EE8DA0> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000001FA16F17390> False
<tensorflow.python.keras.layers.pooling.MaxPo

### 5. Utilize the the model trained on CIFAR 10 (classes 0 to 4) to classify the classes 5 to 9 of CIFAR 10  (Use Transfer Learning) <br>
Achieve an accuracy of more than 85% on test data

In [0]:
# pre process labels
y_train_59 = y_train_59 - 5
y_test_59 = y_test_59 - 5

In [0]:
# one hot encoding
num_class_59 = len(np.unique(y_train_59))

y_train_59 = tf.keras.utils.to_categorical(y_train_59, num_classes=num_class_59)
y_test_59 = tf.keras.utils.to_categorical(y_test_59, num_classes=num_class_59)

In [0]:
# normalizze input
X_train_59 = X_train_59.astype("float32")/255.0
X_test_59 = X_test_59.astype("float32")/255.0

In [0]:
# fit the model by retraining only dense layers 

epochs=10
batch_size = 32

model.fit(X_train_59, y_train_59, validation_data=(X_test_59, y_test_59), batch_size=batch_size, epochs=epochs)

W0916 21:00:07.403886 13952 training.py:1952] Discrepancy between trainable weights and collected trainable weights, did you set `model.trainable` without calling `model.compile` after ?


Train on 25000 samples, validate on 5000 samples
Epoch 1/10
25000/25000 [==============================] - 126s 5ms/sample - loss: 1.1516 - accuracy: 0.5270 - val_loss: 0.9479 - val_accuracy: 0.6354
Epoch 2/10
25000/25000 [==============================] - 130s 5ms/sample - loss: 0.8496 - accuracy: 0.6701 - val_loss: 0.8511 - val_accuracy: 0.6718
Epoch 3/10
25000/25000 [==============================] - 130s 5ms/sample - loss: 0.7362 - accuracy: 0.7165 - val_loss: 0.6912 - val_accuracy: 0.7460
Epoch 4/10
25000/25000 [==============================] - 130s 5ms/sample - loss: 0.6735 - accuracy: 0.7470 - val_loss: 0.6453 - val_accuracy: 0.7622
Epoch 5/10
25000/25000 [==============================] - 133s 5ms/sample - loss: 0.6299 - accuracy: 0.7633 - val_loss: 0.6376 - val_accuracy: 0.7518
Epoch 6/10
25000/25000 [==============================] - 135s 5ms/sample - loss: 0.5917 - accuracy: 0.7774 - val_loss: 0.6011 - val_accuracy: 0.7724
Epoch 7/10
25000/25000 [===========================

In [0]:
score = model.evaluate(X_test_59, y_test_59)
print('\n', 'Test accuracy for Model with Transfer Learning :', score[1]*100)

5000/5000 [==============================] - 4s 709us/sample - loss: 0.5893 - accuracy: 0.7834

 Test accuracy for Model with Transfer Learning : 78.33999991416931


# Text classification using TF-IDF

### 6. Load the dataset from sklearn.datasets

In [0]:
from sklearn.datasets import fetch_20newsgroups

In [0]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

### 7. Training data

In [3]:
twenty_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

### 8. Test data

In [0]:
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

###  a.  You can access the values for the target variable using .target attribute 
###  b. You can access the name of the class in the target variable with .target_names


In [5]:
twenty_train.target

array([1, 1, 3, ..., 2, 2, 2])

In [6]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [7]:
twenty_train.data[0:5]

['From: sd345@city.ac.uk (Michael Collier)\nSubject: Converting images to HP LaserJet III?\nNntp-Posting-Host: hampton\nOrganization: The City University\nLines: 14\n\nDoes anyone know of a good way (standard PC application/PD utility) to\nconvert tif/img/tga files into LaserJet III format.  We would also like to\ndo the same, converting to HPGL (HP plotter) files.\n\nPlease email any response.\n\nIs this the correct group?\n\nThanks in advance.  Michael.\n-- \nMichael Collier (Programmer)                 The Computer Unit,\nEmail: M.P.Collier@uk.ac.city                The City University,\nTel: 071 477-8000 x3769                      London,\nFax: 071 477-8565                            EC1V 0HB.\n',
 "From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\nSubject: help: Splitting a trimming region along a mesh \nOrganization: University Of Kentucky, Dept. of Math Sciences\nLines: 28\n\n\n\n\tHi,\n\n\tI have a problem, I hope some of the 'gurus' can help me solve.\n\n\tBackground of the probl

### 9.  Now with dependent and independent data available for both train and test datasets, using TfidfVectorizer fit and transform the training data and test data and get the tfidf features for both

In [0]:
# use TfIDFVectorizer to create document-term matrices 
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()

In [0]:
twenty_train_dtm = vect.fit_transform(twenty_train.data)

In [0]:
twenty_test_dtm = vect.transform(twenty_test.data)

### 10. Use logisticRegression with tfidf features as input and targets as output and train the model and report the train and test accuracy score

In [0]:
# import and instantiate a logistic regression model
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression()

In [18]:
# train the model using X_train_dtm
logreg.fit(twenty_train_dtm, twenty_train.target)

/usr/local/lib/python3.6/dist-packages/sklearn/linear_model/logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/linear_model/logistic.py:469: FutureWarning: Default multi_class will be changed to 'auto' in 0.22. Specify the multi_class option to silence this warning.
  "this warning.", FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=None, solver='warn', tol=0.0001, verbose=0,
                   warm_start=False)

In [0]:
# make class predictions for X_test_dtm
y_pred_class_log = logreg.predict(twenty_test_dtm)

In [20]:
# calculate accuracy
from sklearn import metrics

metrics.accuracy_score(twenty_test.target, y_pred_class_log)

0.8868175765645806